In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import os
from natsort import natsorted

import scanpy as sc
import seaborn as sns

from scroutines import basicu

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tools.sm_exceptions import ValueWarning
from tqdm import tqdm


import sys
sys.path.insert(0, '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/myvisctx/analysis_multiome/')
import lmm

In [2]:
%%time
outfigdir = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/'
f = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/superdupermegaRNA_hasraw_multiome_IT.h5ad'
adata_raw = sc.read(f)
adata_raw

CPU times: user 869 ms, sys: 9.49 s, total: 10.4 s
Wall time: 10.7 s


AnnData object with n_obs × n_vars = 89287 × 16567
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden', 'time'
    var: 'feature_types'
    layers: 'norm'

In [3]:
f = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/L5IT_labels_gao25_to_yoo25_knn.csv' 
df_lbl = pd.read_csv(f)
df_lbl

,label,conf
GTTAAGCTCCTTGCGT-1-P21a-2023 Multiome-9-0,56_L5 IT CTX Glut_1,0.924669
AAACCGAAGGTCTTGG-1-P21a-2023 Multiome-9-0,62_L5 IT CTX Glut_3,0.729984
AAACCGCGTAAACAAG-1-P21a-2023 Multiome-9-0,64_L5 IT CTX Glut_4,0.562360
AAACGGATCACCATTT-1-P21a-2023 Multiome-9-0,62_L5 IT CTX Glut_3,0.623916
AAATCCGGTGCTTACT-1-P21a-2023 Multiome-9-0,64_L5 IT CTX Glut_4,0.871148
...,...,...
GCTAAGCGTCCGTGAG-1-P21DRb-2023 Multiome-10-0,64_L5 IT CTX Glut_4,0.598713
AAACGTACAGTATGTT-1-P21DRa-2023 Multiome-10-0,62_L5 IT CTX Glut_3,0.593614
GGTTGCATCTGCAACG-1-P21DRa-2023 Multiome-10-0,62_L5 IT CTX Glut_3,0.521387
CGCCAAATCATAGCCG-1-P21DRa-2023 Multiome-10-0,64_L5 IT CTX Glut_4,0.512278


In [4]:
adata = adata_raw[df_lbl.index].copy()
adata.obs = adata.obs.join(df_lbl)
adata.obs
adata

AnnData object with n_obs × n_vars = 1211 × 16567
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden', 'time', 'label', 'conf'
    var: 'feature_types'
    layers: 'norm'

In [5]:
adata.X.data

array([32.,  2.,  1., ..., 18.,  1., 18.], dtype=float32)

In [6]:
adata.obs['Age'].unique()

['P21', 'P21DR']
Categories (2, object): ['P21', 'P21DR']

In [7]:
adata.obs['Sample'].unique()

['P21a', 'P21b', 'P21DRb', 'P21DRa']
Categories (4, object): ['P21DRa', 'P21DRb', 'P21a', 'P21b']

In [8]:
adata.obs['total_counts'].unique()

array([54036., 27254., 26099., ...,  7491.,  6755., 12872.], dtype=float32)

In [9]:
clusters = np.sort(adata.obs['label'].unique())
clusters

array(['56_L5 IT CTX Glut_1', '62_L5 IT CTX Glut_3',
       '63_L5 IT CTX Glut_4', '64_L5 IT CTX Glut_4'], dtype=object)

In [10]:
import time

In [11]:
%%time

obs_fixed1 = 'Age'
obs_fixed2 = None # 'Light'
obs_random = 'Sample'

cluster_col = 'label'

offset = 1e-2
scale = 1e4

for cluster in clusters:
    tag = f"d260303_{cluster.replace('/', '').replace(' ', '_')}"
    output = os.path.join(outfigdir, f'NRDR_DEGs_LMM_yoo25_P21_{tag}.csv')

    adatasub = adata[adata.obs[cluster_col]==cluster]
    genes = adatasub.var.index.values 

    if obs_fixed2 is None:
        obs = adatasub.obs[[obs_fixed1, obs_random]].copy()
    else:
        obs = adatasub.obs[[obs_fixed1, obs_fixed2, obs_random]].copy()
    obs = obs.dropna()
    adatasub = adatasub[obs.index]

    # mat_raw = np.array(adatasub.X.todense())
    # mat_raw = np.array(adatasub.raw.X.todense())
    mat_raw = np.array(adatasub.X.todense())
    
    # ### test
    # adatasub = adatasub[:,:20]
    # genes = genes[:20]
    # mat_raw = mat_raw[:,:20]
    # ### test

    # mat (CP10k norm)
    # mat = mat_raw/adatasub.obs['n_counts'].values.reshape(-1,1)*scale
    mat = mat_raw/adatasub.obs['total_counts'].values.reshape(-1,1)*scale

    res = lmm.run_lmm(mat, genes, obs, obs_fixed1, obs_random, output_csv=output, offset=offset)
    print(output)

(108, 16567) (108, 2)
(108, 14554) (108, 2)
(108, 10131) (108, 2)
1698 ['Tcea1' 'Gm26901' 'Gm28836' ... 'Trub1' 'Fam204a' 'Tmlhe']


100% 10131/10131 [13:54<00:00, 12.13it/s]


11 ['Acvr1c' 'Gk' 'Xist' 'Cmc2' 'Nefm' '4930517O19Rik' 'Cyp11a1' 'Sec14l2'
 'Stac2' 'Pitpnc1' 'Twsg1']
save to csv: /u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_56_L5_IT_CTX_Glut_1.csv
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_56_L5_IT_CTX_Glut_1.csv
(139, 16567) (139, 2)
(139, 14462) (139, 2)
(139, 9998) (139, 2)
1170 ['4732440D04Rik' 'Snhg6' 'Cpa6' ... 'mt-Co3' 'mt-Nd3' 'mt-Nd4']


100% 9998/9998 [11:29<00:00, 14.50it/s]


274 ['Hs6st1' 'Zdbf2' 'Bcs1l' 'Panct2' 'Cdh7' 'Tor3a' 'Rgs4' 'Srp9' 'Sertad4'
 'Tmem141' 'Sec16a' 'BC005624' 'Mrrf' 'Nr6a1os' 'Nr4a2' 'Harbi1'
 'D930015M05Rik' 'Cdan1' 'Gatm' 'Btbd3' 'Banf2' 'Cdk5rap1' 'Trp53inp2'
 '4930405A21Rik' 'Rbl1' '9430021M05Rik' 'Gdap1l1' 'Gmeb2' 'Zgpat' 'Ftsj1'
 'Abcd1' 'Rpl10' 'Xist' 'Pbdc1' 'Itm2a' 'Armcx2' 'A230072E10Rik' 'Gyg'
 '1700027H10Rik' 'Tiparp' '1110032F04Rik' 'Scamp3' 'Kcnn3' 'S100a10'
 'Wars2' 'Cttnbp2nl' 'AI504432' 'Palmd' 'Alg14' 'Abcd3' 'Rrh' 'Gm43192'
 'Gm17501' 'Ddah1' 'Lyn' 'Plekhf2' 'Exosc3' 'Shb' 'E130308A19Rik' 'Prpf4'
 '0610043K17Rik' 'Plpp3' 'Ndc1' 'Gale' 'Eno1' 'Tnfrsf25' 'Vwa1' 'Armc10'
 'Abcf2' 'Smarcd3' 'Qdpr' 'Aasdh' 'Grsf1' 'Dck' 'Ssh1' 'Gm14508' 'Gm40323'
 'Psph' 'Znhit1' 'Gm42456' 'Nyap1' 'Tnrc18' 'Rpa3' '1700111E14Rik'
 'E330009J07Rik' 'Zfp786' 'Wbp1' 'Hdac11' 'Xpc' 'Grip2' 'Gm26911'
 'Bhlhe40' 'Tada3' 'Ccnd2' 'Tmc4' 'Leng9' 'Nat14' 'Fiz1' 'Lig1' 'Gm42372'
 'Dmpk' 'Qpctl' 'Pvr' 'Spred3' 'A030001D20Rik' 'Svip' 'Gm32647' 'Arrb1'

100% 9717/9717 [08:31<00:00, 18.99it/s]


34 ['1700066M21Rik' 'Ddx31' 'Hsd17b12' 'Mamld1' 'Xist' '4921511C10Rik' 'Clk2'
 'Cachd1' 'Ak4' 'Sgsm1' 'Vgf' 'Fosb' 'Snx3' 'Nudt4' 'Gpt2' 'Gm10629'
 'Snupn' 'Arid3b' 'Eif4a1' '1700016P03Rik' 'Dusp14' 'Fam120a' 'Gm29361'
 'Inf2' 'Trib1' 'Arc' 'Phf21b' 'Sh3gl1' 'Rps14' 'Npas4' 'Gda' 'Ina'
 'Neurl1a' 'Dusp5']
save to csv: /u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_63_L5_IT_CTX_Glut_4.csv
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_63_L5_IT_CTX_Glut_4.csv
(795, 16567) (795, 2)
(795, 16015) (795, 2)
(795, 9173) (795, 2)
150 ['Prex2' 'Zdbf2' 'Ikzf2' 'Fn1' 'Neu2' 'Lemd1' 'Btg2' 'Apobec4' 'Rnasel'
 'Ier5' 'Atp1a2' 'Camk1g' 'Ptgds' 'Ntng2' 'Tnfaip6' 'Nr4a2' 'Gm13889'
 'Bdnf' 'Ivd' 'Gatm' 'Dok5' 'Mamld1' 'Xist' 'Glra2' 'Noct' 'Maml3' 'Mme'
 'Tiparp' '4921511C10Rik' 'S100a13' 'Gstm1' 'Egf' 'Gm17501' 'Ptger3'
 'Gm12371' 'Tgfbr1' 'Nr4a3' 'Ak4' 'Stk40' 'Serinc2' 'Gpr3' 'Zfp46' 'Thap3'
 'Rheb' 'Fosl2' 'G

100% 9173/9173 [10:36<00:00, 14.41it/s]


58 ['Zdbf2' 'Neu2' 'Btg2' 'Camk1g' 'Ntng2' 'Dok5' 'Xist' '4921511C10Rik'
 'Gm17501' 'Tgfbr1' 'Nr4a3' 'Ak4' 'Gpr3' 'Thap3' 'Rheb' 'Gm16054' 'Parm1'
 'Sgsm1' 'Rph3a' 'Nptx2' 'Foxp2' 'Mest' 'Tnfrsf23' 'Egr2' 'Gadd45b'
 'Dusp6' 'Vegfc' 'Mast3' 'Gm45435' 'Slc9a5' 'Sntb2' 'Cbfa2t3' 'Abhd4'
 'Egr3' 'Smad3' 'Gm48677' 'Kremen1' 'Per1' 'Camkk1' 'Faap100' 'Pcsk1'
 'Homer1' '4933413L06Rik' 'Frmd6' 'Myh9' 'Ccdc134' 'Phf21b' 'Grasp'
 'Arhgap31' 'Plcxd2' 'Lix1' 'Sik1' 'Plekhh2' 'Kdm5d' 'Eif2s3y' 'Uty'
 'Ddx3y' 'Papss2']
save to csv: /u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_64_L5_IT_CTX_Glut_4.csv
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_64_L5_IT_CTX_Glut_4.csv
CPU times: user 44min 10s, sys: 22.9 s, total: 44min 33s
Wall time: 44min 34s


In [12]:
adata

AnnData object with n_obs × n_vars = 1211 × 16567
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden', 'time', 'label', 'conf'
    var: 'feature_types'
    layers: 'norm'